# 07.1 — Curriculum learning intuition

You don't teach someone calculus on day one. You start with arithmetic, then algebra, then calculus — a *curriculum*. Training the Optimal decoder works the same way: instead of switching every loss and every trainable layer on at once, the model is trained in **stages** that gradually increase in difficulty. Early epochs learn to *reconstruct* the neural signal (an easy, unsupervised task); later epochs learn to *classify* from that representation (the hard task); and the regularization is *annealed* in gradually rather than clamped on from the start. This module is that curriculum, and this notebook is why it exists.

This notebook covers:

* What curriculum learning is, and the human-learning analogy.
* Why a VAE decoder specifically benefits from staged training.
* The three levers the curriculum controls: load, loss weights, freezing.
* KL annealing as the flagship example, shown with the real schedule interpolator.

**Prerequisites:** [05.6 two-stage lifecycle](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb), [06.4 EMA prior normalization](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb).

## Section 1 — What MATLAB does

The Optimal config drives training with a **"Soft Three-Stage Curriculum"** — a set of schedules that vary hyperparameters *per epoch* rather than holding them fixed:

```matlab
% Each of these is a schedule (waypoints), not a constant:
LoadParameters  = cgg_generateLoadParameters_v2(...);   % data augmentation over epochs
LossWeights     = cgg_generateLossWeights_v2(...);      % component weights (incl. KL annealing)
FrozenNetwork   = cgg_setFrozenNetwork_v2(...);         % which subnetworks are trainable

for epoch = 1:numEpochs
    load_now   = cgg_calculateDynamicValue(LoadParameters,  epoch);   % interpolate
    weights_now = cgg_calculateDynamicValue(LossWeights,     epoch);
    frozen_now = cgg_calculateDynamicValue(FrozenNetwork,   epoch);
    % ... train one epoch with these epoch-specific values ...
end
```

The single primitive underneath all three is `cgg_calculateDynamicValue` — piecewise-linear interpolation between waypoints ([07.2](07.2_piecewise_linear_schedules.ipynb)). The port mirrors this: a `Schedule` object holds `ScheduledParameter`s, and `schedule.update(epoch)` re-interpolates them all.

## Section 2 — The Python concepts you need

### 2.1 — Curriculum learning, in one idea

**Start easy, get harder.** Present the model with a simplified version of the task first, then progressively increase difficulty as it becomes competent. The intuition: a model that has already learned *some* structure is far better positioned to learn the hard part than one thrown at the full problem cold. It's the same reason you learn scales before sonatas — early competence scaffolds later competence.

For a neural decoder, "difficulty" has three concrete dials, and the curriculum turns each one over the course of training.

### 2.2 — Why a VAE decoder benefits specifically

Three failure modes that staged training avoids:

* **Reconstruct before you classify.** The encoder must first learn a *useful latent representation* of the neural signal. If you demand classification from epoch 1, the classifier backpropagates through a random, meaningless encoder and drags it toward whatever shortcut fits the labels — a bad basin. Unsupervised reconstruction first ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)) gives the classifier a *real* representation to work with.
* **Anneal the KL, or the latent collapses.** The KL term ([06.2](../06_loss_orchestration/06.2_vae_and_the_elbo.ipynb)) pulls the latent posterior toward a featureless unit Gaussian. Applied at full strength from epoch 1 — before the encoder has learned anything worth keeping — it wins, and the latent becomes pure noise carrying *no information* ("posterior collapse"). Ramping the KL weight up *gradually* lets the encoder first learn structure, then regularizes it. This is the flagship curriculum move (§2.4).
* **Freeze what isn't ready.** Training the classifier on a half-baked encoder, or letting the classifier corrupt the encoder mid-reconstruction, wastes capacity. Freezing subnetworks at the right stages ([07.5](07.5_freeze_unfreeze_curriculum.ipynb)) keeps each part learning only when its inputs are ready.

### 2.3 — The three levers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 4.5))
stages = [("Stage 1\nreconstruct", 0, 5, "#cce4ff"),
          ("Stage 2\nanneal + classify", 5, 11, "#ffe4cc"),
          ("Stage 3\nrefine", 11, 16, "#e6f0d0")]
levers = ["LoadParameters\n(augmentation)", "LossWeights\n(KL annealing)", "FrozenNetwork\n(trainable modules)"]
for i, lever in enumerate(levers):
    y = 2 - i * 0.9
    ax.text(-0.3, y, lever, ha="right", va="center", fontsize=9, fontweight="bold")
    for name, x0, x1, color in stages:
        ax.add_patch(mpatches.Rectangle((x0, y - 0.32), x1 - x0, 0.64,
            facecolor=color, edgecolor="black", lw=1))
for name, x0, x1, color in stages:
    ax.text((x0 + x1) / 2, 2.9, name, ha="center", va="center", fontsize=9.5, fontweight="bold")
    ax.axvline(x0, color="gray", ls=":", lw=0.8)
ax.text(2.5, 1.1, "recon only", ha="center", fontsize=8, style="italic")
ax.text(8, 1.1, "KL ramps 0→1", ha="center", fontsize=8, style="italic", color="crimson")
ax.text(13.5, 1.1, "all active", ha="center", fontsize=8, style="italic")
ax.set_xlim(-3.5, 16.5); ax.set_ylim(-0.2, 3.3); ax.set_xlabel("epoch"); ax.set_yticks([]); ax.set_title("The Soft Three-Stage Curriculum: three levers, varied per epoch")
plt.tight_layout(); plt.show()
print("Three schedules, one timeline: each lever changes what the model is asked to do, when.")

The curriculum is exactly these three per-epoch schedules:

* **LoadParameters** ([07.3](07.3_load_parameters.ipynb)) — the data-loading/augmentation knobs (how much noise, scaling, etc.) presented each epoch.
* **LossWeights** ([07.4](07.4_loss_weights_curriculum.ipynb)) — the per-component weights feeding the multi-objective sum ([06.1](../06_loss_orchestration/06.1_multi_task_losses_overview.ipynb)), including the KL annealing ramp.
* **FrozenNetwork** ([07.5](07.5_freeze_unfreeze_curriculum.ipynb)) — which subnetworks have `requires_grad=True` this epoch.

All three ride on the same interpolation primitive ([07.2](07.2_piecewise_linear_schedules.ipynb)), and the whole thing is traced end-to-end in [07.6](07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb).

### 2.4 — KL annealing, with the real interpolator

In [ ]:
from neural_data_decoding.training.schedules.interpolator import piecewise_anneal_value

# KL weight: OFF for the first 5 epochs (let the encoder learn structure),
# then ramp up to full strength by epoch 10, then hold.
base = 1.0
epoch_waypoints = [5, 10]     # ramp between epoch 5 and epoch 10
magnitudes      = [0.0, 1.0]  # from 0× (off) to 1× (full KL)

print("epoch | KL weight | (encoder freely learns structure, THEN gets regularized)")
for e in range(1, 13):
    kl_w = piecewise_anneal_value(base, epoch_waypoints, magnitudes, e)
    print(f"  {e:2}  |   {kl_w:.2f}    |  {'·' * int(kl_w * 20)}")
print("→ KL stays 0 while the latent learns, then ramps in — avoiding posterior collapse.")

That ramp is the whole point. For the first five epochs the KL weight is `0`, so the encoder is free to learn an informative latent with no pressure to look Gaussian. Then the weight climbs to full strength, gently regularizing the *already-learned* representation instead of crushing a random one. (The exact ramp shape has a one-epoch MATLAB parity quirk — the last step snaps rather than lands smoothly — dissected in [07.2](07.2_piecewise_linear_schedules.ipynb).)

## Section 3 — The `neural_data_decoding` implementation

The schedule subsystem lives in `training/schedules/`. The `Schedule` object holds the scheduled parameters and re-interpolates them each epoch:

In [ ]:
from neural_data_decoding.training.schedules.schedule import Schedule
import inspect

print("Schedule public methods:")
for name in ("update", "current", "names"):
    if hasattr(Schedule, name):
        sig = inspect.signature(getattr(Schedule, name))
        print(f"  Schedule.{name}{sig}")
print()
print(inspect.getdoc(Schedule.update))

Things to spot:

* `Schedule.update(epoch)` re-interpolates every parameter for the new epoch; `Schedule.current(name)` reads a parameter's current value — the training loop calls `update` once per epoch, then reads the levers it needs.
* Each parameter is a `ScheduledParameter` wrapping waypoints + the `piecewise_anneal_value` primitive ([07.2](07.2_piecewise_linear_schedules.ipynb)).
* The presets (the "Soft Three-Stage Curriculum" and friends) live in `schedules/library.py`; they're built into a `Schedule` by `schedules/factory.py` from config ([07.3](07.3_load_parameters.ipynb)–[07.5](07.5_freeze_unfreeze_curriculum.ipynb) cover each lever).
* The loop applies the levers per epoch: freeze changes toggle `requires_grad` ([07.5](07.5_freeze_unfreeze_curriculum.ipynb)), LR writes go to `param_group["lr"]` ([05.4](../05_training_loop/05.4_learning_rate_scheduling.ipynb)), loss weights feed the aggregator ([06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)) — all driven by the one schedule.

## Section 4 — Hands-on exercises

### Exercise 4.1 — a three-stage weight

Build a schedule for a weight that is `0` for epochs 1–3, ramps to `1.0` by epoch 6, then *drops back* to `0.5` by epoch 10 (a rise-then-decay curriculum). Print it.

In [ ]:
# Reveal:
for e in range(1, 12):
    v = piecewise_anneal_value(1.0, [3, 6, 10], [0.0, 1.0, 0.5], e)
    print(f"  epoch {e:2}: {v:.2f}  {'#' * int(v * 20)}")
print("→ three waypoints → two ramps: up (epochs 3–6), then down (epochs 6–10).")

### Exercise 4.2 — why not just train everything at once?

There's no code to write — reason it through. If you set the KL weight to full strength from epoch 1 on an untrained encoder, what happens to the latent, and why does that hurt classification? (Answer below.)

In [ ]:
# Reveal:
print('''Full KL from epoch 1 → the KL term dominates before the encoder learns
anything → the posterior is pushed to the unit Gaussian prior → the latent
carries NO information about the input ("posterior collapse"). The classifier
then reads a noise latent and can't decode. Annealing the KL in gradually lets
the encoder build an informative latent FIRST, so there's real structure to
regularize (and to classify from). Curriculum > cold start.''')

## Section 5 — Common errors

### The latent collapses to noise (posterior collapse)

The KL weight was too high, too early (§2.2). Anneal it in with a schedule ([07.4](07.4_loss_weights_curriculum.ipynb)) — 0 for a warmup, then ramp up.

### Classification never improves

The encoder may be getting corrupted by the classifier before it has learned to reconstruct. Check the freeze schedule ([07.5](07.5_freeze_unfreeze_curriculum.ipynb)) and the two-stage handoff ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)) — reconstruct first.

### The schedule doesn't change anything

`schedule.update(epoch)` must be called each epoch, and the loop must actually *read* the current values ([07.2 §3](07.2_piecewise_linear_schedules.ipynb)). A schedule built but never updated holds its initial values forever.

### Hyperparameters tuned for constant training behave differently under a curriculum

Weights, LR, and freeze states that worked as constants may not transfer — the curriculum changes them over time. Tune against the *schedule*, not a fixed point.

### Waypoints out of order

`epoch_points` must be monotonically increasing ([07.2](07.2_piecewise_linear_schedules.ipynb)). Unsorted waypoints make the segment search pick the wrong interval and the interpolation goes haywire.

## Section 6 — Further reading

- [Curriculum Learning (Bengio et al., 2009)](https://dl.acm.org/doi/10.1145/1553374.1553380) — the paper that framed staged training for neural networks.
- [Generating Sentences from a Continuous Space (Bowman et al., 2016)](https://arxiv.org/abs/1511.06349) — introduced KL annealing to fight VAE posterior collapse.
- [`src/neural_data_decoding/training/schedules/`](../../src/neural_data_decoding/training/schedules/) — the schedule subsystem.

Next up: **[07.2 piecewise-linear schedules](07.2_piecewise_linear_schedules.ipynb)** — the interpolation primitive under every curriculum lever, and its MATLAB off-by-one quirk.